### A simple sentence transformer embedding

In [ ]:
import shutil
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
def load_all_pdfs(pdf_directory: str):
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} pdf files")

    all_documents = []
    for pdf_file in pdf_files:
        print(f"processing {pdf_file}")
        loader = PyPDFLoader(str(pdf_file))
        documents = loader.load()
        for doc in documents:
            doc.metadata["source_file"] = pdf_file.name
            doc.metadata["file_type"] = "pdf"
        all_documents.extend(documents)
        print(f"loaded {len(documents)} pages from {pdf_file.name}")

    return all_documents

all_pdf = load_all_pdfs("../policies")


In [ ]:
def remove_header_footer(text, header_lines=1, footer_lines=1):
    lines = text.split('\n')
    # Remove empty lines from edges first
    lines = [l for l in lines if l.strip()]
    # Skip header and footer lines
    trimmed = lines[header_lines: len(lines) - footer_lines if footer_lines else None]
    return '\n'.join(trimmed)

for page in all_pdf:
    page.page_content = remove_header_footer(page.page_content, header_lines=2, footer_lines=2)


def merge_pages_by_file(documents: list) -> list:
    """One Document per PDF so splitting can span page breaks (section 3 often spans pages)."""
    groups: dict[str, list] = {}
    order: list[str] = []
    for doc in documents:
        key = doc.metadata.get("source_file") or doc.metadata.get("source") or "default"
        if key not in groups:
            groups[key] = []
            order.append(key)
        groups[key].append(doc)
    merged: list[Document] = []
    for key in order:
        batch = sorted(groups[key], key=lambda d: d.metadata.get("page", 0))
        text = "\n\n".join(d.page_content for d in batch)
        meta = {**batch[0].metadata, "merged_page_count": len(batch)}
        merged.append(Document(page_content=text, metadata=meta))
    return merged


all_pdf = merge_pages_by_file(all_pdf)

In [ ]:
all_pdf

In [ ]:
def split_documents(documents, chunk_size: int = 2000, chunk_overlap: int = 200):
    # Prefer breaks before numbered sub-clauses (3.1., 3.2., …) so one chunk
    # keeps more of "Leave Entitlements" together; reduces mid-3.1 truncation.
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=[
            "\n\n",
            r"\n(?=\s*\d+\.\d+\.\s)",
            "\n",
            ". ",
            " ",
        ],
        is_separator_regex=True,
    )
    split_docs = splitter.split_documents(documents)
    print(f"split {len(split_docs)} chunks from {len(documents)} pages")
    return split_docs

In [ ]:
chunks = split_documents(all_pdf)

In [ ]:
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Same model dimension as `sentence-transformers/all-MiniLM-L6-v2`
lc_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

persist_directory = "allMiniLM-vector-store"
collection_name = "allMiniLM-policy-v3"

# # Chroma persists across runs. Re-using the same folder + re-embedding with new
# # chunking can leave OLD chunks in the collection so hits still look "truncated".
# # Delete the store whenever you change split size, merge strategy, or PDFs.
# shutil.rmtree(persist_directory, ignore_errors=True)

lc_vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=lc_embeddings,
    persist_directory=persist_directory,
    collection_name=collection_name,
)

# lambda_mult closer to 1.0 favors relevance; lower values diversify (can skip the best chunk).
lc_retriever = lc_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8, "fetch_k": 24, "lambda_mult": 0.75},
)

In [ ]:
ans = lc_retriever.invoke("Leave Entitlements policy")


In [ ]:
# [ans.page_content for ans in ans]
ans